In [ ]:
import sys

print(sys.executable)

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# ============================================================
# PROJECT SETUP
# Hỗ trợ Google Colab và VS Code/Jupyter
# ============================================================

REPO_URL = "https://github.com/pnga1206/market-risk-measurement.git"
BRANCH = "feature/notebook"
REPO_NAME = "market-risk-measurement"

if "google.colab" in sys.modules:
    # Google Colab
    ROOT = Path("/content") / REPO_NAME

    if not ROOT.exists():
        subprocess.run(
            [
                "git", "clone",
                "--branch", BRANCH,
                "--single-branch",
                REPO_URL,
                str(ROOT)
            ],
            check=True
        )
    else:
        subprocess.run(
            ["git", "-C", str(ROOT), "fetch", "origin"],
            check=True
        )
        subprocess.run(
            ["git", "-C", str(ROOT), "checkout", BRANCH],
            check=True
        )
        subprocess.run(
            ["git", "-C", str(ROOT), "pull", "--ff-only", "origin", BRANCH],
            check=True
        )

else:
    # VS Code / Jupyter local
    ROOT = Path.cwd()

    if ROOT.name == "notebooks":
        ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Cài thư viện tự động khi chạy trên Colab
if "google.colab" in sys.modules:
    requirements_file = ROOT / "requirements.txt"

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirements_file)
    ])

print("Project root:", ROOT)
print("Environment:", "Google Colab" if "google.colab" in sys.modules else "Local")
print("Python:", sys.executable)

In [ ]:
import numpy as np
import pandas as pd
import scipy
import arch
import matplotlib.pyplot as plt

from pathlib import Path

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Arch:", arch.__version__)

In [ ]:
# ============================================================
# DATA PATH
# ============================================================

DATA_DIR = ROOT / "data" / "main"
OUTPUT_TABLES = ROOT / "outputs" / "tables"
OUTPUT_FIGURES = ROOT / "outputs" / "figures"

OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

print("Data folder:", DATA_DIR)
print("Output tables:", OUTPUT_TABLES)
print("Output figures:", OUTPUT_FIGURES)

In [ ]:
print("Files in data/main:")

for file in sorted(DATA_DIR.iterdir()):
    print("-", file.name)

In [ ]:
# ============================================================
# LOAD PORTFOLIO DATA
# ============================================================

returns_path = DATA_DIR / "portfolio_returns.csv"

returns = pd.read_csv(returns_path)

returns["date"] = pd.to_datetime(returns["date"])
returns = returns.sort_values("date").reset_index(drop=True)

print("Shape:", returns.shape)
print("Columns:")
print(list(returns.columns))

print("\nDate range:")
print(returns["date"].min(), "→", returns["date"].max())

returns.head()

In [ ]:
# ============================================================
# DATA QUALITY CHECK
# ============================================================

print("Missing values:")
print(returns.isna().sum())

print("\nDuplicate dates:", returns["date"].duplicated().sum())

print("\nData types:")
print(returns.dtypes)

In [ ]:
# ============================================================
# PORTFOLIO RETURN SUMMARY
# ============================================================

portfolio_returns = returns["portfolio_return"].dropna()

print(portfolio_returns.describe())

In [ ]:
# ============================================================
# PREPARE RETURN SERIES
# ============================================================

portfolio_return_series = (
    returns
    .set_index("date")["portfolio_return"]
    .dropna()
    .sort_index()
)

print("Number of observations:", len(portfolio_return_series))
print("Start:", portfolio_return_series.index.min())
print("End:", portfolio_return_series.index.max())

In [ ]:
# ============================================================
# IMPORT VaR MODELS
# ============================================================

from src.var.var_models import (
    calculate_historical_var,
    calculate_parametric_var,
    calculate_monte_carlo_var,
    run_rolling_var
)

from src.config import CONF_LEVEL, MC_SIMULATIONS, RANDOM_SEED

print("Confidence level:", CONF_LEVEL)
print("Monte Carlo simulations:", MC_SIMULATIONS)
print("Random seed:", RANDOM_SEED)

In [ ]:
# ============================================================
# VaR 97.5% - 1 DAY
# ============================================================

var_historical = calculate_historical_var(
    portfolio_return_series,
    conf_level=CONF_LEVEL
)

var_parametric = calculate_parametric_var(
    portfolio_return_series,
    conf_level=CONF_LEVEL
)

var_monte_carlo = calculate_monte_carlo_var(
    portfolio_return_series,
    conf_level=CONF_LEVEL,
    n_sims=MC_SIMULATIONS,
    seed=RANDOM_SEED
)

var_summary = pd.DataFrame({
    "Method": [
        "Historical",
        "Parametric",
        "Monte Carlo"
    ],
    "VaR": [
        var_historical,
        var_parametric,
        var_monte_carlo
    ]
})

var_summary["VaR (%)"] = var_summary["VaR"] * 100

var_summary

In [ ]:
# ============================================================
# ROLLING VaR 97.5%
# Window = 250 trading days
# ============================================================

ROLLING_WINDOW = 250

rolling_var = run_rolling_var(
    portfolio_return_series,
    window=ROLLING_WINDOW,
    conf_level=CONF_LEVEL
)

print("Shape:", rolling_var.shape)
print("Start:", rolling_var.index.min())
print("End:", rolling_var.index.max())

rolling_var.head()

In [ ]:
# ============================================================
# ROLLING VaR SUMMARY
# ============================================================

rolling_var.describe()

In [ ]:
# ============================================================
# ROLLING VaR FIGURE
# ============================================================

plt.figure(figsize=(12, 6))

plt.plot(
    rolling_var.index,
    rolling_var["VaR_Historical"] * 100,
    label="Historical VaR"
)

plt.plot(
    rolling_var.index,
    rolling_var["VaR_Parametric"] * 100,
    label="Parametric VaR",
    linestyle="--"
)

plt.plot(
    rolling_var.index,
    rolling_var["VaR_MonteCarlo"] * 100,
    label="Monte Carlo VaR",
    linestyle=":"
)

plt.title("Rolling VaR 97.5% – Portfolio VN-Index & S&P 500")
plt.xlabel("Date")
plt.ylabel("VaR (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

figure_path = OUTPUT_FIGURES / "rolling_var_975.png"
plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Đã lưu:", figure_path)

In [ ]:
# ============================================================
# SAVE ROLLING VaR RESULTS
# ============================================================

rolling_var_path = OUTPUT_TABLES / "rolling_var_975.csv"

rolling_var.to_csv(rolling_var_path)

print("Đã lưu:", rolling_var_path)

In [ ]:
# ============================================================
# IMPORT GARCH MODEL
# ============================================================

from src.garch.garch_model import calculate_garch_var_es_rolling

print("GARCH model imported successfully.")

In [ ]:
# ============================================================
# ROLLING GARCH(1,1) + VaR + ES 97.5%
# ============================================================

garch_results = calculate_garch_var_es_rolling(
    portfolio_return_series,
    window=ROLLING_WINDOW,
    conf_level=CONF_LEVEL
)

print("Shape:", garch_results.shape)
print("Start:", garch_results.index.min())
print("End:", garch_results.index.max())

garch_results.head()

In [ ]:
# ============================================================
# GARCH + ES SUMMARY
# ============================================================

garch_results.describe()

In [ ]:
# ============================================================
# LATEST GARCH VaR & ES
# ============================================================

garch_results.tail(1)

In [ ]:
# ============================================================
# SAVE GARCH RESULTS
# ============================================================

garch_path = OUTPUT_TABLES / "garch_var_es_rolling_975.csv"

garch_results.to_csv(garch_path)

print("Đã lưu:", garch_path)

In [ ]:
# ============================================================
# CHECK CURRENT OUTPUTS
# ============================================================

print("Output tables:")

for file in sorted(OUTPUT_TABLES.iterdir()):
    print("-", file.name)

print("\nOutput figures:")

for file in sorted(OUTPUT_FIGURES.iterdir()):
    print("-", file.name)

In [ ]:
# ============================================================
# LATEST VaR RESULTS
# ============================================================

latest_var = rolling_var.tail(1).copy()

latest_var["VaR_Historical_%"] = latest_var["VaR_Historical"] * 100
latest_var["VaR_Parametric_%"] = latest_var["VaR_Parametric"] * 100
latest_var["VaR_MonteCarlo_%"] = latest_var["VaR_MonteCarlo"] * 100

latest_var[
    [
        "VaR_Historical_%",
        "VaR_Parametric_%",
        "VaR_MonteCarlo_%"
    ]
]

In [ ]:
# ============================================================
# LATEST GARCH + ES RESULTS
# ============================================================

latest_garch = garch_results.tail(1).copy()

latest_garch["GARCH_Volatility_%"] = (
    latest_garch["GARCH_Volatility"] * 100
)

latest_garch["VaR_GARCH_%"] = (
    latest_garch["VaR_GARCH"] * 100
)

latest_garch["ES_97_5_%"] = (
    latest_garch["ES_97_5"] * 100
)

latest_garch[
    [
        "GARCH_Volatility_%",
        "VaR_GARCH_%",
        "ES_97_5_%"
    ]
]